Step 1: Exploring the data

In [2]:
import pandas as pd

In [3]:
df=pd.read_csv('dataset/archive (5)/Resume/Resume.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ID           2484 non-null   int64
 1   Resume_str   2484 non-null   str  
 2   Resume_html  2484 non-null   str  
 3   Category     2484 non-null   str  
dtypes: int64(1), str(3)
memory usage: 52.3 MB


In [5]:
df.head(3)

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [6]:
df.shape

(2484, 4)

In [7]:
df['Category'].unique()

<ArrowStringArray>
[                    'HR',               'DESIGNER', 'INFORMATION-TECHNOLOGY',
                'TEACHER',               'ADVOCATE',   'BUSINESS-DEVELOPMENT',
             'HEALTHCARE',                'FITNESS',            'AGRICULTURE',
                    'BPO',                  'SALES',             'CONSULTANT',
          'DIGITAL-MEDIA',             'AUTOMOBILE',                   'CHEF',
                'FINANCE',                'APPAREL',            'ENGINEERING',
             'ACCOUNTANT',           'CONSTRUCTION',       'PUBLIC-RELATIONS',
                'BANKING',                   'ARTS',               'AVIATION']
Length: 24, dtype: str

In [8]:
df['Category'].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df['Resume_str'].duplicated().sum()

np.int64(2)

In [11]:
df[df['Resume_str'].duplicated(keep=False)]


,ID,Resume_str,Resume_html,Category
1490,19147603,FINANCE OFFICER Professional ...,"<div class=""fontsize fontface vmargins hmargin...",FINANCE
1509,28398216,FINANCE OFFICER Professional ...,"<div class=""fontsize fontface vmargins hmargin...",FINANCE
2444,16850314,STOREKEEPER II Professional Sum...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2483,37473139,STOREKEEPER II Professional Sum...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION


In [12]:
df = df.drop_duplicates(subset='Resume_str', keep='first').reset_index(drop=True)


In [13]:
df.shape

(2482, 4)

In [14]:
df['Resume_str'][0]

"         HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss pr

In [15]:
print(repr(df['Resume_str'].iloc[1200]))


"         SENIOR CONSULTANT           Experience      Senior Consultant  ,     09/2015   to   Current     Company Name   –   City  ,   State      Manage the relationship between CVS Health Med D enrollment operations and EGS (Expert Global Solutions), a.  vendor contracted to process member centric requests and operational processes with 230+ employees.  Engage.  with site directors, operations managers, HR, trainers, workforce consultants, and supervisors to strategically.  resolve workflow and deliverable issues.  Ensure continued service delivery and quality satisfaction from EGS and a successful working relationship between CVS and EGS.  Travel to two main sites bi-monthly during Med D's annual enrollment period to ensure successful training execution.  Set service expectations for each line of business.  Successfully brought up a vendor site with 100+ employees with a 2-month period, including access to all systems, training and escalations.  Raised quality from 70% to an average 

Step 2 - A: cleaning 

In [16]:
import re

In [17]:
def clean_text(text: str)->str:
    #remove null/control bytes
    text=re.sub(r"[\x00-\x08\x0b\x0c\x0e\x0f]", "", text)

    #normalize dashes
    text=re.sub(r"[‐-‒–—―−\uff0d]+", "-",text)

    # Collapse repeated spaces/tabs, without removing newlines
    text = re.sub(r"[^\S\r\n]+", " ", text)
    #collapse multiple newlines and removing spaces around them
    text = re.sub(r"[ \t\r]*\n[ \t\r\n]*", "\n", text)

    return text.strip()


In [18]:
df['Resume_clean']=df['Resume_str'].apply(clean_text)

In [19]:
len(df['Resume_clean'])

2482

In [20]:
count=0
for i in range(len(df['Resume_clean'])):
    if len(df['Resume_clean'][i])<20:
        count+=1
        print(i)
    else:
        continue
print("count of junk or empty resumes: ",count)


656
count of junk or empty resumes:  1


In [21]:
df = df.drop(index=656).reset_index(drop=True)


#step 2 - B: Tokenization 

In [22]:
import spacy

In [23]:
nlp = spacy.load("en_core_web_md")

In [24]:
import string

def preprocessing(text: str):
    doc = nlp(text, disable=["parser", "ner"])
    filtered = []
    for token in doc:
        if token.is_stop or token.is_punct or token.is_space:
            continue
        lemma = token.lemma_.lower().strip(string.punctuation)
        if len(lemma) < 2 or not any(c.isalpha() for c in lemma):
            continue
        filtered.append(lemma)
    return filtered

preprocessing(df['Resume_clean'][1200])

['managing',
 'consultant',
 'summary',
 'highly',
 'accomplished',
 'management',
 'consultant',
 'senior',
 'business',
 'analyst',
 'verifiable',
 'track',
 'record',
 'manage',
 'complex',
 'strategy',
 'project',
 'exceed',
 'client',
 'expectation',
 'demonstrate',
 'skill',
 'business',
 'process',
 'management',
 'process',
 'redesign',
 'specialize',
 'end',
 'end',
 'business',
 'process',
 'management',
 'lifecycle',
 'extensive',
 'experience',
 'integration',
 'implementation',
 'organizational',
 'transformational',
 'effort',
 'public',
 'financial',
 'services',
 'sectors',
 'designing',
 'process',
 'system',
 'improvement',
 'increase',
 'productivity',
 'reduce',
 'cost',
 'strong',
 'interpersonal',
 'skill',
 'highly',
 'adept',
 'manage',
 'broad',
 'stakeholder',
 'community',
 'support',
 'development',
 'cohesive',
 'strategic',
 'vision',
 'disparate',
 'group',
 'skills',
 'business',
 'process',
 'improvement',
 'redesign',
 'agile',
 'scrum',
 'sdlc',
 'bus

In [25]:
df['tokens']=df['Resume_clean'].apply(preprocessing)

In [26]:
df['tokens'].apply(len).describe()

count    2481.00000
mean      573.49738
std       258.00497
min        67.00000
25%       464.00000
50%       536.00000
75%       661.00000
max      3473.00000
Name: tokens, dtype: float64

In [27]:
count=0
check=[]
for i in range(len(df['tokens'])):
    if len(df['tokens'][i])<100:
        count+=1
        print(i)
        check.append(df['Category'][i])
    else:
        continue
print("count of junk or empty resumes: ",count)
check

130
153
1038
1048
1101
1929
count of junk or empty resumes:  6


['DESIGNER', 'DESIGNER', 'SALES', 'SALES', 'SALES', 'CONSTRUCTION']

In [28]:
df['tokens'][153]

['floral',
 'designer',
 'skills',
 'billings',
 'cash',
 'handling',
 'cashier',
 'creativity',
 'customer',
 'service',
 'magic',
 'pick',
 'pos',
 'experience',
 'jan',
 'current',
 'company',
 'city',
 'state',
 'floral',
 'designer',
 'jan',
 'company',
 'city',
 'state',
 'designer',
 'jan',
 'company',
 'city',
 'state',
 'assign',
 'errand',
 'duty',
 'customer',
 'service',
 'design',
 'work',
 'event',
 'set',
 'magic',
 'city',
 'floral',
 'billings',
 'mt',
 'customer',
 'service',
 'miscellaneous',
 'assign',
 'duty',
 'floral',
 'designer',
 'delivery',
 'driver',
 'jan',
 'jan',
 'company',
 'city',
 'state',
 'assign',
 'duty',
 'education',
 'training',
 'work',
 'floral',
 'design',
 'certificate',
 'fall',
 'range',
 'community',
 'college',
 'range',
 'community',
 'college',
 'work',
 'floral',
 'design',
 'certificate',
 'spring',
 'associates',
 'horticulture',
 'fall',
 'range',
 'community',
 'college',
 'horticulture',
 'spring',
 'colorado',
 'state',
 'unive

Step 3: Building the model 

In [29]:
from sklearn.ensemble import RandomForestClassifier
from  sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression

In [30]:
from sklearn.model_selection import train_test_split

In [31]:
x_train,x_test,y_train,y_test=train_test_split(df['tokens'],df['Category'],test_size=0.2,stratify=df['Category'],random_state=42)

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

# your tokens are already cleaned/lemmatized, so join them back into strings
x_train_str = x_train.apply(' '.join)
x_test_str = x_test.apply(' '.join)

vectorizer = TfidfVectorizer()
x_train_vec = vectorizer.fit_transform(x_train_str)
x_test_vec = vectorizer.transform(x_test_str)


Baseline models

In [33]:
model_1=LogisticRegression(class_weight='balanced')
model_1.fit(x_train_vec,y_train)
y_pred_1=model_1.predict(x_test_vec)
print(classification_report(y_test,y_pred_1))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.57      0.83      0.68        24
              ADVOCATE       0.42      0.42      0.42        24
           AGRICULTURE       0.70      0.54      0.61        13
               APPAREL       0.60      0.16      0.25        19
                  ARTS       0.46      0.29      0.35        21
            AUTOMOBILE       0.57      0.57      0.57         7
              AVIATION       0.86      0.78      0.82        23
               BANKING       0.88      0.65      0.75        23
                   BPO       0.67      0.50      0.57         4
  BUSINESS-DEVELOPMENT       0.49      0.79      0.60        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.81      0.77      0.79        22
            CONSULTANT       0.43      0.13      0.20        23
              DESIGNER       0.81      0.81      0.81        21
         DIGITAL-MEDIA       0.79      

In [34]:
model_2=RandomForestClassifier(class_weight='balanced', random_state=42)
model_2.fit(x_train_vec,y_train)
y_pred_2=model_2.predict(x_test_vec)
print(classification_report(y_test,y_pred_2))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.54      0.92      0.68        24
              ADVOCATE       0.78      0.58      0.67        24
           AGRICULTURE       1.00      0.54      0.70        13
               APPAREL       0.86      0.32      0.46        19
                  ARTS       0.50      0.10      0.16        21
            AUTOMOBILE       1.00      0.14      0.25         7
              AVIATION       0.77      0.87      0.82        23
               BANKING       0.86      0.52      0.65        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.49      0.71      0.58        24
                  CHEF       0.82      0.75      0.78        24
          CONSTRUCTION       0.75      0.82      0.78        22
            CONSULTANT       0.67      0.26      0.38        23
              DESIGNER       0.78      0.86      0.82        21
         DIGITAL-MEDIA       0.69      

c:\Users\myous\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\myous\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\myous\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [35]:
model_3=KNeighborsClassifier()
model_3.fit(x_train_vec,y_train)
y_pred_3=model_3.predict(x_test_vec)
print(classification_report(y_test,y_pred_3))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.51      0.88      0.65        24
              ADVOCATE       0.27      0.58      0.37        24
           AGRICULTURE       0.57      0.31      0.40        13
               APPAREL       0.33      0.21      0.26        19
                  ARTS       0.46      0.29      0.35        21
            AUTOMOBILE       0.25      0.29      0.27         7
              AVIATION       0.69      0.39      0.50        23
               BANKING       0.63      0.52      0.57        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.46      0.75      0.57        24
                  CHEF       0.86      0.75      0.80        24
          CONSTRUCTION       0.61      0.77      0.68        22
            CONSULTANT       0.36      0.22      0.27        23
              DESIGNER       0.71      0.71      0.71        21
         DIGITAL-MEDIA       0.65      

N-gram and vocabulary-pruning experiments (both eventually superseded by hyperparameter tuning below, kept as documented negative/neutral results)

In [36]:
vectorizer_bi = TfidfVectorizer(ngram_range=(1, 2))
x_train_vec_bi = vectorizer_bi.fit_transform(x_train_str)
x_test_vec_bi = vectorizer_bi.transform(x_test_str)

model_bi = LogisticRegression(class_weight='balanced')
model_bi.fit(x_train_vec_bi, y_train)
y_pred_bi = model_bi.predict(x_test_vec_bi)
print(classification_report(y_test, y_pred_bi, zero_division=0))


                        precision    recall  f1-score   support

            ACCOUNTANT       0.51      0.83      0.63        24
              ADVOCATE       0.39      0.46      0.42        24
           AGRICULTURE       0.78      0.54      0.64        13
               APPAREL       0.50      0.16      0.24        19
                  ARTS       0.62      0.24      0.34        21
            AUTOMOBILE       1.00      0.29      0.44         7
              AVIATION       0.85      0.74      0.79        23
               BANKING       0.88      0.65      0.75        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.48      0.88      0.62        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.82      0.82      0.82        22
            CONSULTANT       1.00      0.13      0.23        23
              DESIGNER       0.82      0.86      0.84        21
         DIGITAL-MEDIA       0.79      

In [37]:
vectorizer_pruned = TfidfVectorizer(min_df=2, max_df=0.85)
x_train_vec_pruned = vectorizer_pruned.fit_transform(x_train_str)
x_test_vec_pruned = vectorizer_pruned.transform(x_test_str)

model_pruned = LogisticRegression(class_weight='balanced')
model_pruned.fit(x_train_vec_pruned, y_train)
y_pred_pruned = model_pruned.predict(x_test_vec_pruned)
print(classification_report(y_test, y_pred_pruned, zero_division=0))



                        precision    recall  f1-score   support

            ACCOUNTANT       0.57      0.83      0.68        24
              ADVOCATE       0.42      0.42      0.42        24
           AGRICULTURE       0.70      0.54      0.61        13
               APPAREL       0.67      0.21      0.32        19
                  ARTS       0.46      0.29      0.35        21
            AUTOMOBILE       0.57      0.57      0.57         7
              AVIATION       0.86      0.78      0.82        23
               BANKING       0.94      0.65      0.77        23
                   BPO       0.67      0.50      0.57         4
  BUSINESS-DEVELOPMENT       0.51      0.79      0.62        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.82      0.82      0.82        22
            CONSULTANT       0.38      0.13      0.19        23
              DESIGNER       0.81      0.81      0.81        21
         DIGITAL-MEDIA       0.79      

In [38]:
len(vectorizer.get_feature_names_out())

31856

In [39]:
len(vectorizer_pruned.get_feature_names_out())

15006

Hyperparameter tuning (GridSearchCV + cross-validation) - final model selection

In [40]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

param_grid = {
    'tfidf__min_df': [1, 2, 3],
    'tfidf__max_df': [0.85, 0.9, 1.0],
    'clf__C': [0.01, 0.1, 1, 10]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1
)
grid_search.fit(x_train_str, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV macro-F1:", grid_search.best_score_)


Best params: {'clf__C': 10, 'tfidf__max_df': 0.85, 'tfidf__min_df': 2}
Best CV macro-F1: 0.6287575864869215


In [41]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(x_test_str)
print(classification_report(y_test, y_pred_best, zero_division=0))


                        precision    recall  f1-score   support

            ACCOUNTANT       0.59      0.83      0.69        24
              ADVOCATE       0.52      0.62      0.57        24
           AGRICULTURE       0.78      0.54      0.64        13
               APPAREL       0.50      0.37      0.42        19
                  ARTS       0.47      0.43      0.45        21
            AUTOMOBILE       0.75      0.43      0.55         7
              AVIATION       0.85      0.74      0.79        23
               BANKING       0.84      0.70      0.76        23
                   BPO       0.50      0.25      0.33         4
  BUSINESS-DEVELOPMENT       0.54      0.79      0.64        24
                  CHEF       0.84      0.67      0.74        24
          CONSTRUCTION       0.78      0.82      0.80        22
            CONSULTANT       0.60      0.26      0.36        23
              DESIGNER       0.84      0.76      0.80        21
         DIGITAL-MEDIA       0.67      

In [42]:
pipeline_rf = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'tfidf__min_df': [1, 2],
    'tfidf__max_df': [0.85, 1.0],
    'clf__n_estimators': [200, 400,600,1000],
    'clf__max_depth': [None, 30],
    'clf__class_weight': ['balanced', 'balanced_subsample']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search_rf = GridSearchCV(
    pipeline_rf, param_grid_rf, cv=cv, scoring='f1_macro', n_jobs=-1
)
grid_search_rf.fit(x_train_str, y_train)

print("Best params:", grid_search_rf.best_params_)
print("Best CV macro-F1:", grid_search_rf.best_score_)

best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(x_test_str)
print(classification_report(y_test, y_pred_rf, zero_division=0))


Best params: {'clf__class_weight': 'balanced', 'clf__max_depth': 30, 'clf__n_estimators': 600, 'tfidf__max_df': 0.85, 'tfidf__min_df': 2}
Best CV macro-F1: 0.6099781732646521
                        precision    recall  f1-score   support

            ACCOUNTANT       0.51      0.88      0.65        24
              ADVOCATE       0.88      0.58      0.70        24
           AGRICULTURE       1.00      0.54      0.70        13
               APPAREL       0.75      0.32      0.44        19
                  ARTS       1.00      0.05      0.09        21
            AUTOMOBILE       1.00      0.29      0.44         7
              AVIATION       0.78      0.78      0.78        23
               BANKING       0.78      0.61      0.68        23
                   BPO       1.00      0.25      0.40         4
  BUSINESS-DEVELOPMENT       0.50      0.75      0.60        24
                  CHEF       0.83      0.83      0.83        24
          CONSTRUCTION       0.76      0.86      0.81   

In [43]:
import joblib
joblib.dump(grid_search.best_estimator_, 'model/resume_classifier.joblib')


['Resume ATS system/model/resume_classifier.joblib']

Step 4: PDF ingestion

In [44]:
import pdfplumber 

In [45]:
def extract_text_from_pdf(pdf_path:str)->str:
    with pdfplumber.open(pdf_path) as pdf:
        text=[]
        for i in range(len(pdf.pages)):
            text.append(pdf.pages[i].extract_text() or '')
    return '\n'.join(text)
    

In [46]:
test_sample=extract_text_from_pdf("dataset/archive (5)/data/data/AVIATION/10189110.pdf")

In [47]:
cleaned = clean_text(test_sample)
tokens = preprocessing(cleaned)
tokens_str = ' '.join(tokens)
grid_search.best_estimator_.predict([tokens_str])


array(['AVIATION'], dtype=object)

Step 5: Entity extraction (NER) - hybrid: spaCy for DATE, HuggingFace transformer (scored) for PERSON/ORG/LOCATION

In [48]:
def extract_entities(text: str) -> dict:
    doc = nlp(text, disable=["parser"])
    entities = {"PERSON": [], "ORG": [], "GPE": [], "DATE": []}
    for ent in doc.ents:
        if ent.label_ in entities:
            entities[ent.label_].append(ent.text)
    return entities


In [49]:
df['Resume_clean'][0]

"HR ADMINISTRATOR/MARKETING ASSOCIATE\nHR ADMINISTRATOR Summary Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management. Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service. Highlights Focused on customer satisfaction Team management Marketing savvy Conflict resolution techniques Training and development Skilled multi-tasker Client relations specialist Accomplishments Missouri DOT Supervisor Training Certification Certified by IHG in Customer Loyalty and Marketing by Segment Hilton Worldwide General Manager Training Certification Accomplished Trainer for cross server hospitality systems such as Hilton OnQ , Micros Opera PMS , Fidelio OPERA Reservation System (ORS) , Holidex Completed courses and seminars in customer service, sales strategies, inventory control, loss prevention, safety, time management, leadership and performance assessment. Experience HR A

In [50]:
extract_entities(extract_text_from_pdf("C:/Users/myous/Downloads/MY_CV_QA.pdf"))

{'PERSON': ['Mohamed Ahmed Hassanein Youssef',
  'Docker',
  'Docker',
  'Streamlit',
  'Jupyter Notebook',
  'F1',
  'Git',
  'Ultralytics YOLOv8',
  'Streamlit',
  'Docker',
  'Docker'],
 'ORG': ['Machine Learning & Computer Vision',
  'Machine Learning',
  'Computer Vision',
  'Python',
  'SQL',
  'TensorFlow',
  'PyTorch',
  'Machine Learning & Deep Learning',
  'PyTorch',
  'XGBoost',
  'LightGBM\nComputer Vision: OpenCV',
  'Transfer Learning',
  'Model Deployment & MLOps',
  'Data Analysis & Visualization',
  'Pandas',
  'EDA',
  'Feature Engineering',
  'Seaborn\nTools & Platforms',
  'GitHub',
  'Google Colab\nSoft Skills: Problem Solving',
  'Communication, Teamwork',
  'Time Management\nLanguages: Arabic (Native',
  'National Telecommunication Institute',
  'NTI',
  'CNN',
  '• Evaluated',
  'Google Colab',
  'EDA',
  'Brain Tumor MRI Classification',
  'EfficientNet',
  'TensorFlow/Keras',
  'Google Colab',
  'YOLO',
  'Kaggle',
  'Google Colab',
  'EfficientNet',
  'CNN',


In [51]:
from transformers import pipeline as hf_pipeline

ner_pipeline = hf_pipeline(
    "ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=0
)




c:\Users\myous\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6738.19it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [52]:
def extract_entities_scored(text: str, threshold: float = 0.8) -> dict:
    results = ner_pipeline(text, stride=50)
    entities = {"PER": [], "ORG": [], "LOC": [], "MISC": []}
    for ent in results:
        if ent["score"] >= threshold and ent["entity_group"] in entities:
            entities[ent["entity_group"]].append((ent["word"], round(float(ent["score"]), 3)))
    return entities


In [53]:
extract_entities_scored(df['Resume_clean'][0].strip(),0.80)

{'PER': [],
 'ORG': [('Hospitality and Customer Service Management', 0.982),
  ('IHG', 0.807),
  ('Hilton Worldwide', 0.907),
  ('Hilton OnQ', 0.968),
  ('Micros Opera PM', 0.968),
  ('Fidelio OPERA Reservation System', 0.953),
  ('ORS', 0.91),
  ('Holidex', 0.861),
  ('Chamber of Commerce', 0.997),
  ('Jefferson College', 0.922),
  ('State Business Administration', 0.989),
  ('State', 0.853)],
 'LOC': [],
 'MISC': [('I', 0.867)]}

In [54]:
extract_entities_scored(extract_text_from_pdf("C:/Users/myous/Downloads/MY_CV_QA.pdf"),0.95)

{'PER': [('Mohamed Ahmed Hassan', 0.997)],
 'ORG': [('Machine Learning & Computer Vision', 0.985),
  ('Deep Learning', 0.989),
  ('Computer Vision', 0.986),
  ('Ke', 0.955),
  ('E', 0.967),
  ('CNN', 0.993),
  ('Telecom Egypt', 0.963),
  ('Ke', 0.972),
  ('##icientNet', 0.973),
  ('Deep Learning Institute', 0.977)],
 'LOC': [('Do', 0.995),
  ('##ha', 0.981),
  ('Qatar', 0.999),
  ('Egypt', 0.973),
  ('Egypt', 0.966),
  ('Alexandria', 0.999),
  ('Egypt', 1.0)],
 'MISC': [('Arabic', 0.995), ('English', 0.997)]}

Step 6: Skills & Education extraction (PhraseMatcher)

In [55]:
import sys
sys.path.append("Resume ATS system")
from skills_keywords import ALL_SKILLS, AMBIGUOUS_SKILLS, EDUCATION_LEVELS, canonicalize_skill
from spacy.matcher import PhraseMatcher

non_ambiguous = [s for s in ALL_SKILLS if s not in AMBIGUOUS_SKILLS]
skill_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
skill_matcher.add("SKILL", [nlp.make_doc(s) for s in non_ambiguous])

ambiguous_matcher = PhraseMatcher(nlp.vocab)
ambiguous_matcher.add("AMBIGUOUS_SKILL", [nlp.make_doc(s) for s in AMBIGUOUS_SKILLS])

edu_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
edu_matcher.add("EDUCATION", [nlp.make_doc(e) for e in EDUCATION_LEVELS])


In [56]:
def collect_matches(matcher, doc):
    results = []
    for match_id, start, end in matcher(doc):
        span = doc[start:end]
        results.append({"text": span.text, "start_char": span.start_char, "end_char": span.end_char})
    return results


def remove_overlapping_matches(matches):
    matches = sorted(
        matches,
        key=lambda item: (item["start_char"], -(item["end_char"] - item["start_char"]))
    )
    selected = []
    for current in matches:
        overlaps = any(
            current["start_char"] < saved["end_char"] and current["end_char"] > saved["start_char"]
            for saved in selected
        )
        if not overlaps:
            selected.append(current)
    return selected


In [57]:
def extract_skills_and_education(text: str) -> dict:
    doc = nlp(text)

    raw_skill_matches = collect_matches(skill_matcher, doc) + collect_matches(ambiguous_matcher, doc)
    skill_matches = remove_overlapping_matches(raw_skill_matches)

    canonical_skills = []
    seen = set()
    for m in skill_matches:
        canonical = canonicalize_skill(m["text"])
        if canonical.lower() not in seen:
            seen.add(canonical.lower())
            canonical_skills.append(canonical)

    edu_matches = remove_overlapping_matches(collect_matches(edu_matcher, doc))
    education = []
    seen_edu = set()
    for m in edu_matches:
        if m["text"].lower() not in seen_edu:
            seen_edu.add(m["text"].lower())
            education.append(m["text"])

    return {"skills": canonical_skills, "education": education}


In [58]:
extract_skills_and_education(clean_text(extract_text_from_pdf("C:/Users/myous/Downloads/MY_CV_QA.pdf")))

{'skills': ['Artificial Intelligence',
  'Machine Learning',
  'Computer Vision',
  'Deep Learning',
  'Feature Engineering',
  'FastAPI',
  'Docker',
  'Python',
  'SQL',
  'TensorFlow',
  'PyTorch',
  'Predictive Modeling',
  'Transfer Learning',
  'Scikit-learn',
  'Keras',
  'XGBoost',
  'LightGBM',
  'OpenCV',
  'Model Deployment',
  'MLOps',
  'Data Analysis',
  'NumPy',
  'Pandas',
  'Exploratory Data Analysis',
  'Matplotlib',
  'Seaborn',
  'Git',
  'GitHub',
  'Jupyter Notebook',
  'Google Colab',
  'Problem Solving',
  'Communication',
  'Teamwork',
  'Time Management',
  'Convolutional Neural Network',
  'Hyperparameter Tuning',
  'EfficientNet',
  'Scheduling',
  'Ultralytics',
  'YOLO',
  'REST API',
  'SQLite',
  'Data Science'],
 'education': ['Bachelor of Engineering', 'Professional Certificate']}

In [59]:
extract_skills_and_education(df['Resume_clean'][0].strip())

{'skills': ['Marketing',
  'Customer Service',
  'Hospitality',
  'Customer Satisfaction Score',
  'Conflict Resolution',
  'Training and Development',
  'Sales',
  'Inventory Control',
  'Loss Prevention',
  'Time Management',
  'Leadership',
  'Compensation',
  'Labor Relations',
  'Documentation',
  'Statistics',
  'Employee Relations',
  'ICD-9',
  'CPT',
  'Medical Billing',
  'Data Analysis',
  'Budgeting',
  'Accounting',
  'Payroll',
  'Purchasing',
  'Swift'],
 'education': ['Certification', 'High School Diploma']}

Step 7: Job matching

In [60]:
vectorizer = grid_search.best_estimator_.named_steps["tfidf"]


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def _skills_for_matching(raw_text: str) -> tuple[str, set[str]]:
    """
    Extract skills for job/resume matching, reduced from raw text - strips
    out narrative language ("we are seeking...") and job-history verbs that
    don't reflect actual required competencies. Classification still uses
    the full preprocessed text; this reduction is specific to matching.

    Returns both a joined string (for TF-IDF cosine similarity) and a
    lowercase set (for skill-coverage calculation), computed from a single
    extraction pass rather than extracting twice.
    """
    skills = extract_skills_and_education(clean_text(raw_text))["skills"]
    return ' '.join(skills), {s.lower() for s in skills}


def rank_resumes_for_job(job_description: str, resumes: dict[str, str]) -> list[dict]:
    """
    resumes: {identifier (e.g. filename): RAW resume text}

    Returns a list of dicts sorted by cosine_similarity descending, each with:
      - candidate: the identifier
      - cosine_similarity: overall similarity between extracted skill sets.
        Can be misleadingly low for a candidate with many skills beyond
        what the job needs - cosine similarity is diluted by "extra"
        content even when every required skill is present (verified: a
        candidate with only the 5 exact required skills scored 1.0, the
        same candidate's real 43-skill resume scored 0.389, despite having
        all 5 required skills - having more skills than needed actively
        lowers this score). skill_coverage below doesn't have that problem.
      - skill_coverage: fraction of the job's required skills the candidate
        actually has (0.0-1.0) - unaffected by how many extra skills the
        candidate also has.
    """
    job_text, job_skill_set = _skills_for_matching(job_description)
    job_vec = vectorizer.transform([job_text])

    names = list(resumes.keys())
    resume_texts = []
    resume_skill_sets = []
    for text in resumes.values():
        t, s = _skills_for_matching(text)
        resume_texts.append(t)
        resume_skill_sets.append(s)

    resume_vecs = vectorizer.transform(resume_texts)
    cosine_scores = cosine_similarity(job_vec, resume_vecs)[0]

    results = []
    for name, cos_score, resume_skills in zip(names, cosine_scores, resume_skill_sets):
        coverage = len(job_skill_set & resume_skills) / len(job_skill_set) if job_skill_set else 0.0
        results.append({
            "candidate": name,
            "cosine_similarity": float(cos_score),
            "skill_coverage": coverage,
        })

    return sorted(results, key=lambda r: r["cosine_similarity"], reverse=True)


In [92]:
description = ''' 
# Junior AI Engineer – Machine Learning & Computer Vision

## Job Summary

We are looking for a motivated Junior AI Engineer to join our technology team and assist in developing practical machine learning and computer vision solutions.

The ideal candidate should have a strong foundation in Python, machine learning, deep learning, and image processing. The role is suitable for a recent Computer Science or Computer Engineering graduate with hands-on experience building, evaluating, and deploying machine learning models through academic, internship, or personal projects.

## Key Responsibilities

* Develop and train machine learning and deep learning models using Python.
* Build computer vision solutions for image classification and object detection.
* Prepare image and structured datasets for model training.
* Perform data cleaning, preprocessing, feature engineering, and data augmentation.
* Train convolutional neural networks and transfer-learning models.
* Use pretrained architectures such as EfficientNet and YOLO.
* Evaluate models using accuracy, precision, recall, F1-score, and confusion matrices.
* Apply techniques to reduce overfitting and handle class imbalance.
* Tune model parameters and compare different machine learning approaches.
* Develop inference pipelines for processing new images.
* Build REST API endpoints to serve trained models.
* Deploy machine learning applications using FastAPI and Docker.
* Create simple model demonstrations using Streamlit.
* Document experiments, model results, and technical decisions.
* Manage source code and project versions using Git and GitHub.
* Work with other developers to integrate AI models into web or mobile applications.
* Research suitable machine learning methods for business and user problems.

## Required Qualifications

* Bachelor’s degree in Computer Science, Computer Engineering, Artificial Intelligence, Data Science, or a related field.
* Zero to two years of professional or internship experience.
* Strong programming skills in Python.
* Basic knowledge of SQL and relational databases.
* Good understanding of machine learning and deep learning concepts.
* Hands-on experience with TensorFlow, Keras, or PyTorch.
* Experience with scikit-learn, Pandas, and NumPy.
* Knowledge of convolutional neural networks and transfer learning.
* Familiarity with computer vision and image preprocessing.
* Basic experience with OpenCV.
* Understanding of model training, validation, testing, and evaluation.
* Familiarity with REST APIs and model deployment.
* Ability to communicate technical results clearly.
* Strong problem-solving, teamwork, and time-management skills.

## Preferred Qualifications

* Experience with image classification projects.
* Experience training object-detection models using YOLO.
* Familiarity with EfficientNet or other pretrained neural-network architectures.
* Experience using data augmentation to improve model generalization.
* Understanding of class imbalance, weighted loss, and weighted sampling.
* Experience building APIs with FastAPI.
* Familiarity with Docker and containerized model deployment.
* Experience building machine learning interfaces with Streamlit.
* Familiarity with cloud deployment platforms such as Render.
* Experience using Jupyter Notebook or Google Colab.
* A GitHub portfolio containing machine learning or computer vision projects.

## Technical Skills

* Python
* SQL
* Machine Learning
* Deep Learning
* Computer Vision
* Data Analysis
* Data Preprocessing
* Feature Engineering
* Predictive Modeling
* Model Evaluation
* Hyperparameter Tuning
* CNN
* Convolutional Neural Networks
* Transfer Learning
* Image Classification
* Object Detection
* Image Preprocessing
* Data Augmentation
* TensorFlow
* Keras
* PyTorch
* Scikit-learn
* XGBoost
* LightGBM
* OpenCV
* YOLO
* EfficientNet
* Pandas
* NumPy
* Matplotlib
* Seaborn
* FastAPI
* REST API
* Docker
* Streamlit
* Git
* GitHub
* Jupyter Notebook
* Google Colab

## Example Tasks

* Train an image-classification model to identify different product categories.
* Develop an object-detection model that identifies objects from webcam or video input.
* Improve model performance through augmentation and transfer learning.
* Compare TensorFlow and PyTorch models using relevant evaluation metrics.
* Package a trained model into an API using FastAPI.
* Containerize and deploy the API using Docker.
* Add a confidence threshold to prevent unreliable predictions.
* Create a Streamlit demonstration for testing the model.

## Success Measures

* Model accuracy and F1-score.
* Reliability of predictions on unseen data.
* Quality and readability of Python code.
* Reproducibility of model experiments.
* API response speed and stability.
* Successful integration of models into applications.
* Clear documentation of datasets, experiments, and results.
* Ability to complete assigned tasks within project deadlines.
'''

In [93]:
resume=extract_text_from_pdf("C:/Users/myous/Downloads/MY_CV_QA.pdf")

In [ ]:
x=1430
print(df['Category'][x])
rank_resumes_for_job(description, {df['ID'][x]: df['Resume_clean'][x]})
